# Exploratory Data Analysis — COVID-19 Brazil

# 1. Imports

In [ ]:
import pandas
import numpy
import matplotlib.pyplot as plt
from pandas import DataFrame
from IPython.display import Markdown, display

# 2. Data Preparation

This notebook is self-contained: the cleaning pipeline is re-applied here rather than depending on `data_quality.ipynb` having been run first. This guarantees reproducibility — the EDA can be executed independently at any time.

## 2.1 Load Raw Data

CSV has no native null type. Values recorded as `"NA"`, `""`, or `"Importados/Indefinidos"` are read by pandas as plain strings — invisible to `isna()` and unaffected by interpolation. Replacing them with `numpy.nan` at load time makes them proper missing values from the start.

In [ ]:
dataset = pandas.read_csv("data/caso_full.csv")

dataset = dataset.replace(
    ["<NA>", "NA", "NaN", "nan", "null", "", "Importados/Indefinidos"],
    numpy.nan
)

display(Markdown(f"**Rows loaded:** {len(dataset):,}"))

## 2.2 Cleaning Pipeline

Four decisions are applied in sequence. Order matters — each step depends on the previous one.

**`place_type == "state"`** — The dataset contains both city-level and state-level rows for the same dates. State rows already aggregate all cities within them, so keeping both granularities would double-count every case. Since the forecasting model operates at state level, the EDA must use the same granularity.

**`is_last == True`** — The dataset preserves the full history of secretariat corrections: the same state + date combination appears multiple times as numbers are revised over time. Only the most recent revision is the authoritative value. Using any other version means analyzing stale data.

**`sort_values(["state", "date"])`** — Interpolation estimates missing values from neighboring rows. Those neighbors must be the chronologically adjacent days for the same state. Sorting by state then date guarantees this — without it, interpolation would draw from arbitrary rows.

**`where(~group["is_repeated"])`** — `is_repeated = True` means the secretariat published no bulletin that day, so `new_confirmed` is forced to `0` by convention — it does not mean zero new cases. The `where` call turns those zeros into `NaN`, making them eligible for interpolation. Keeping the `0` would teach the model that cases periodically vanish.

**`interpolate(method="linear")`** — Linear interpolation estimates each missing day from its neighbors. Linear is appropriate for daily data with short gaps (typically 1–2 days on weekends). Forward fill would reproduce the same zero-on-no-publication behavior being corrected; spline or polynomial would overfit single-day gaps.

**`.clip(lower=0)`** — Negative values arise when a secretariat reclassifies cases between municipalities. The model cannot learn from a negative case count. Clipping to zero is the least harmful treatment: it does not drop the row (which would create a gap) or interpolate (the value is known, just physically impossible).

In [ ]:
df = dataset[
    (dataset["place_type"] == "state") &
    (dataset["is_last"] == True)
].copy().sort_values(["state", "date"])

def clean_state_series(group):
    group = group.copy()
    group["new_confirmed"] = (
        group["new_confirmed"]
        .where(~group["is_repeated"])
        .interpolate(method="linear")
        .clip(lower=0)
    )
    return group

df = pandas.concat([
    clean_state_series(group)
    for _, group in df.groupby("state")
])

df["date"] = pandas.to_datetime(df["date"])

display(Markdown(f"**Records after cleaning:** {len(df):,}"))
display(Markdown(f"**Date range:** {df['date'].min().date()} \u2192 {df['date'].max().date()}"))
display(Markdown(f"**States:** {df['state'].nunique()}"))
df.head()